In [20]:
%%capture
!pip install biopython openpyxl
import os


# Runs a subset of the model layers to test that the notebook runs in CI, but the output will be incorrect.
FAST_CI_MODE: bool = os.environ.get("FAST_CI_MODE", False)

In [21]:
import glob
import gzip
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from Bio import SeqIO
from sklearn.metrics import auc, roc_auc_score, roc_curve

In [22]:
%%capture
EXPERIMENTAL_1b_CHECKPOINT: bool = False
MODEL_SIZE = "1b"  # also try 7b if you have a GPU with more than 32GB of memory
# Define checkpoint path
if EXPERIMENTAL_1b_CHECKPOINT and MODEL_SIZE == "1b":
    from bionemo.core.data.load import load

    # This is a new 1b checkpoint that has been fine-tuned on BF16 hardware. It should be able to handle FP8 as well.
    #  this line will download the checkpoint from NGC to your $HOME/.cache/bionemo directory and return the path.
    #  alternatively you can use `CHECKPOINT_PATH=$(download_bionemo_data evo2/1b-8k-bf16:1.0)`  to do the same on the
    #  command line.
    checkpoint_path = load("evo2/1b-8k-bf16:1.0")
else:
    checkpoint_path = Path(f"nemo2_evo2_{MODEL_SIZE}_8k")

    # Check if the directory does not exist or is empty
    if not checkpoint_path.exists() or not any(checkpoint_path.iterdir()):
        !evo2_convert_to_nemo2 --model-path hf://arcinstitute/savanna_evo2_{MODEL_SIZE}_base --model-size {MODEL_SIZE} --output-dir nemo2_evo2_{MODEL_SIZE}_8k
    else:
        print("Checkpoint directory is not empty. Skipping command.")

In [23]:
def check_fp8_support():
    """Check if FP8 is supported on the current GPU.

    FP8 requires compute capability 8.9+ (Ada Lovelace/Hopper architecture or newer).
    """
    if not torch.cuda.is_available():
        return False, "CUDA not available"

    device_props = torch.cuda.get_device_properties(0)
    compute_capability = f"{device_props.major}.{device_props.minor}"
    device_name = device_props.name

    # FP8 is supported on compute capability 8.9+ (Ada Lovelace/Hopper architecture)
    is_supported = (device_props.major > 8) or (
        device_props.major == 8 and device_props.minor >= 9
    )

    return (
        is_supported,
        f"Device: {device_name}, Compute Capability: {compute_capability}",
    )

In [24]:
fp8_supported, gpu_info = check_fp8_support()
print(f"FP8 Support: {fp8_supported}")
print(gpu_info)


FP8 Support: False
Device: NVIDIA A100-SXM4-80GB, Compute Capability: 8.0


In [25]:
# Note: If FP8 is not supported, you may want to disable it in the model config
# The Evo2 config has 'use_fp8_input_projections: True' by default

if FAST_CI_MODE:
    model_subset_option = "--num-layers 4 --hybrid-override-pattern SDH*"
else:
    model_subset_option = ""

fp8_option = "--fp8" if fp8_supported else ""

In [26]:
# Define output directories for prediction results
folder_name="CLPD1"
output_dir = Path(f"{folder_name}_fasta_files")
output_dir.mkdir(parents=True, exist_ok=True)

# Save reference and variant sequences to FASTA
ref_fasta_path = output_dir / f"{folder_name}_reference.fasta"
var_fasta_path = output_dir / f"{folder_name}_variants.fasta"

predict_ref_dir = output_dir / "reference_predictions"
predict_var_dir = output_dir / "variant_predictions"
predict_ref_dir.mkdir(parents=True, exist_ok=True)
predict_var_dir.mkdir(parents=True, exist_ok=True)

In [27]:
# Update predict commands to run on the full dataset
predict_ref_command = (
    f"predict_evo2 --fasta {ref_fasta_path} --ckpt-dir {checkpoint_path} "
    f"--output-dir {predict_ref_dir} --model-size {MODEL_SIZE} --tensor-parallel-size 1  {model_subset_option} "
    f"--pipeline-model-parallel-size 1 --context-parallel-size 1 --output-log-prob-seqs {fp8_option}"
)

predict_var_command = (
    f"predict_evo2 --fasta {var_fasta_path} --ckpt-dir {checkpoint_path} "
    f"--output-dir {predict_var_dir} --model-size {MODEL_SIZE} --tensor-parallel-size 1 {model_subset_option} "
    f"--pipeline-model-parallel-size 1 --context-parallel-size 1 --output-log-prob-seqs {fp8_option}"
)

In [28]:
%%capture
print(f"Running command: {predict_ref_command}")
!{predict_ref_command}


In [29]:
%%capture
print(f"Running command: {predict_var_command}")
!{predict_var_command}

In [30]:
# Find and load prediction files
ref_pred_files = glob.glob(os.path.join(predict_ref_dir, "predictions__rank_*.pt"))
var_pred_files = glob.glob(os.path.join(predict_var_dir, "predictions__rank_*.pt"))

# Load sequence ID maps (maps sequence ID -> prediction index)
with open(os.path.join(predict_ref_dir, "seq_idx_map.json"), "r") as f:
    ref_seq_idx_map = json.load(f)
with open(os.path.join(predict_var_dir, "seq_idx_map.json"), "r") as f:
    
    var_seq_idx_map = json.load(f)

# Load predictions
ref_preds = torch.load(ref_pred_files[0])
var_preds = torch.load(var_pred_files[0])

In [31]:
ref_preds

# the fasta files need to have the same name on the header 

{'log_probs_seqs': tensor([-5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966,
         -5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966,
         -5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966, -5.4966,
         -5.4966, -5.4966, -5.4966, -5.4966]),
 'seq_idx': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27])}

In [32]:
# next, calculate change in likelihoods
ref_log_probs = []
var_log_probs = []
# Assuming `Gene_interest_df` is a DataFrame that needs to be created based on the context of the notebook
# Create a placeholder DataFrame for demonstration purposes
Gene_interest_df = pd.DataFrame({
    "ref_fasta_name": list(ref_seq_idx_map.keys()),
    "var_fasta_name": list(var_seq_idx_map.keys())  # Ensure matching lengths
})

for _, row in Gene_interest_df.iterrows():
    ref_name = row["ref_fasta_name"]
    var_name = row["var_fasta_name"]
    ref_log_probs.append(ref_preds["log_probs_seqs"][ref_seq_idx_map[ref_name]].item())
    var_log_probs.append(var_preds["log_probs_seqs"][var_seq_idx_map[var_name]].item())

Gene_interest_df["ref_log_probs"] = ref_log_probs
Gene_interest_df["var_log_probs"] = var_log_probs
# Ideally, the probability of a broken variant is lower than a good one. So a bad var - good ref is negative.
Gene_interest_df["evo2_delta_score"] = Gene_interest_df["var_log_probs"] - Gene_interest_df["ref_log_probs"]
Gene_interest_df.head()

,ref_fasta_name,var_fasta_name,ref_log_probs,var_log_probs,evo2_delta_score
0,HAP10_Freq.27,HAP10_Freq.27,-5.496636,-5.478323,0.018314
1,HAP11_Freq.31,HAP11_Freq.31,-5.496636,-5.473191,0.023446
2,HAP12_Freq.32,HAP12_Freq.32,-5.496636,-5.495887,0.000749
3,HAP13_Freq.33,HAP13_Freq.33,-5.496636,-5.491743,0.004893
4,HAP14_Freq.39,HAP14_Freq.39,-5.496636,-5.474360,0.022276


In [33]:
# save the results to a file using the tag MODEL_SIZE and evo2_delta_score
output_file = f"{output_dir}/evo2_delta_score_{MODEL_SIZE}.csv"
Gene_interest_df.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")
# Load the data

Results saved to CLPD1_fasta_files/evo2_delta_score_1b.csv
